# 05 — Machine-Learning Spatial Interpolation with Urban Canopy Parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/05_ml_interpolation_ucp.en.ipynb)

**Learning objective**: learn an alternative to geostatistical kriging (notebook 04) for turning sparse station
temperature observations into a continuous city-wide map — training a Random Forest (or OLS) regression model on
**Urban Canopy Parameters (UCPs)**, such as building density, building height, and impervious/vegetated land
fraction, extracted at each station location, and then predicting temperature pixel-by-pixel across the LCZ grid
using those same parameters. By the end of this notebook you will be able to run `lcz_interp_map_plus` to produce
an ML-based temperature surface with an accompanying uncertainty map and feature-importance ranking, and use
`lcz_interp_eval_plus` to cross-validate its accuracy — directly comparable to the kriging accuracy from notebook 04.

## Two philosophies of spatial interpolation

Notebook 04 covered **kriging**, a geostatistical method that predicts a value at an unobserved location as a
weighted average of nearby observations, where the weights come from a fitted **variogram** — a model of how
similar two points are as a function of the distance between them. Kriging's core assumption is **spatial
autocorrelation**: things close together tend to be alike, and that similarity decays with distance in a way that
can be captured by a single distance-decay function (plus, in LCZ4py's implementation, an LCZ-class drift term).
This works well when station density is reasonably high and the underlying temperature field varies smoothly in
space — exactly the conditions under which the Tobler's-first-law intuition (Tobler, 1970) holds.

This notebook covers a fundamentally different approach: **regression against urban form**. Instead of asking "how
far is this pixel from a station?", we ask "what does this pixel's *local urban fabric* look like, and how does
temperature respond to that fabric at the stations we do have?" The features are **Urban Canopy Parameters (UCPs)**
— quantities like built-up surface fraction, building height, building volume, and population density, drawn from
the Global Human Settlement Layer (GHSL; Pesaresi & Politis, 2023) and companion morphological datasets. A model
(Random Forest by default) learns the function `temperature ~ f(UCPs)` from the stations, and that function is then
applied to every pixel's own UCP values — regardless of how far that pixel is from the nearest station.

This distinction matters practically. Urban climate theory going back to Oke's canyon-geometry work, and formalized
in the Local Climate Zone scheme (Stewart & Oke, 2012), holds that near-surface air temperature is strongly shaped
by local urban morphology: sky view factor, surface materials, building density, and vegetation fraction drive the
urban heat island effect largely independent of what is happening a few kilometers away. If that relationship is
strong and consistent, a model that has *seen* how temperature responds to built-up fraction and building height at
a handful of stations can generalize to any pixel whose UCP values resemble those training conditions — even one
far from every station. Kriging, in contrast, degrades exactly where you need it most: in the gaps of a sparse
network, its predictions revert toward the global mean regardless of urban form, because distance-decay is its only
lever.

**When to prefer which approach**:

- **Kriging (notebook 04)** — dense, well-distributed station networks; smooth spatial gradients (e.g. large-scale
  regional cooling toward a coastline); when you need a properly quantified spatial covariance structure (kriging
  variance) and don't need to reason about *why* a location is hot or cold.
- **RF + UCP (this notebook)** — sparse or clustered networks, especially with large station-free gaps in
  structurally distinct neighborhoods; strong urban-form heterogeneity (a mix of dense high-rise, low-rise suburbs,
  and parks) where the *cause* of spatial variation is morphological rather than purely geographic; when
  feature-importance / interpretability (which UCP drives temperature the most?) is itself a research question.

The honest way to decide which wins for *your* city and station network is not to pick a favorite in the abstract,
but to run both cross-validations — `lcz_interp_eval` (notebook 04) and `lcz_interp_eval_plus` (this notebook) — on
the same data and compare the resulting RMSE/MAE/R². We do exactly that at the end of this notebook.

## Install LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Reference data: Berlin LCZ map + station observations

As in the rest of the `local/` notebook series, we use Berlin's LCZ map and the bundled `lcz_data_berlin.csv`
station dataset (hourly air temperature, column `airT`, station identifier column `station`). Keeping the same
city and CSV across notebooks 01–08 lets you compare methods on identical data.

In [2]:
import pandas as pd
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Berlin")
print(map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df["date"] = pd.to_datetime(df["date"])
df.head()

06:33:19 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_2ed1dcfe644b4c6b.tif


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01,albrecht,8.04725,52.444594,13.348607
3,806407,2019-01-01,bamberger,8.27166,52.496494,13.337552
4,806408,2019-01-01,baruth,8.20000,52.061300,13.499700


## `lcz_interp_map_plus` — Random Forest regression on Urban Canopy Parameters

`lcz_interp_map_plus(x, data_frame, var, station_id, ml_model="rf", sp_res=..., ucp_variables=..., n_estimators=...,
process_ghsl=..., process_wumpod=..., process_vegetation=..., process_directional=..., ...)` trains an ML model
(`"rf"` = Random Forest, `"ols"` = Multiple Linear Regression) on UCP values extracted at each station, then applies
the fitted model pixel-by-pixel across a north-up prediction grid derived from the LCZ map's bounds.

Key parameters used below, and why:

- `ml_model="rf"` — Random Forest handles non-linear temperature/urban-form relationships and gives us both feature
  importances and prediction uncertainty (standard deviation across trees) for free.
- `sp_res=1000.0` — a coarse 1&nbsp;km output grid. UCP downloads and pixel-wise prediction scale with grid cell
  count, so a coarser grid keeps this runnable interactively; production runs would use something finer
  (e.g. 100&nbsp;m, matching notebook 04).
- `process_ghsl=True` with `process_wumpod=False`, `process_vegetation=False`, `process_directional=False` — this
  restricts the downloaded UCP feature set to the four GHSL layers (built-up surface fraction, building height,
  built-up volume, population density; Pesaresi & Politis, 2023), skipping the WUMPOD morphological rasters,
  vegetation/impervious layers, and 8-direction roughness parameters that `lcz_get_ucp` can otherwise fetch. This
  keeps the download small and fast while still capturing the core drivers of the urban heat island — built-up
  density and building height.
- `n_estimators=50` — a small forest. Fewer trees than the function's default (`200`) trains faster; with only a
  handful of Berlin stations and 4 features, 50 trees is already more than enough to stabilize predictions.
- `year=2019, month=1, day=1` — the CSV spans 700 days; with `tp_res="day"` the function would otherwise fit one
  model per day. Restricting to a single day via the function's built-in date filters keeps this demo interactive;
  drop these arguments (or widen the range) for a full multi-day production run.

The function downloads GHSL urban form rasters cropped to the LCZ map's footprint, extracts their values at each
station's coordinates as training features, fits the model per time step (or per `by=` group), and writes a
multi-band GeoTIFF (`result.path`) of predicted temperature. It also writes a companion uncertainty raster
(`result.uncertainty_path`) and returns `result.feature_importance` — which UCP mattered most for each time step's
prediction.

**Caveat — this cell can be slow.** `lcz_interp_map_plus` downloads real GHSL raster tiles over the network on
first run (cached afterward under `lcz4r_cache/`). If this cell does not complete promptly in your environment, it
is fine to skip execution here and run it yourself locally — the code below is signature-accurate and will produce
a valid `LCZInterpResult` when run.

In [3]:
from LCZ4py.local.lcz_interp_map_plus import lcz_interp_map_plus

result = lcz_interp_map_plus(
    map_path,
    df,
    var="airT",
    station_id="station",
    ml_model="rf",
    sp_res=1000.0,
    tp_res="day",
    n_estimators=50,
    process_ghsl=True,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=False,
    isave=False,
    year=2019, month=1, day=1,  # restrict to a single day to keep this demo fast
)

if result is not None:
    print("Output raster:", result.path)
    print("Model:", result.model_type, "| stations:", result.n_stations,
          "| features:", result.n_features, "| groups:", len(result.groups))
    print("Uncertainty raster:", result.uncertainty_path)
else:
    print("No result — see log messages above (e.g. too few stations per group).")

06:33:22 - LCZ4py.local.lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...


06:33:22 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:33:22 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


06:33:22 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:33:22 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676


...

06:36:10 - LCZ4py.local.lcz_interp_map_plus - INFO - Features: ['pop'] (1 total)


06:36:12 - LCZ4py.local.lcz_interp_map_plus - INFO - RF cross-val R² = -14112.974 ± 28214.997  (n=11, features=1)


06:36:12 - LCZ4py.local.lcz_interp_map_plus - INFO - Group '2019-01-01 00:00:00' RF top-5 features: [('pop', '1.000')]


06:36:12 - LCZ4py.local.lcz_interp_map_plus - INFO - ML interpolation complete: 1 band(s), method=rf


Output raster: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpfjeh03uz.tif
Model: rf | stations: 23 | features: 1 | groups: 1
Uncertainty raster: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpf66zn9v2.tif


### Interpreting the output

`result` is an `LCZInterpResult` with:

- **`path`** — a multi-band GeoTIFF, one band per time step (or `by=` group), each band a predicted temperature
  surface across every pixel of the Berlin LCZ grid.
- **`uncertainty_path`** — a parallel GeoTIFF of prediction uncertainty (for Random Forest: the standard deviation
  of predictions across the individual trees). High uncertainty flags pixels whose UCP combination is unlike
  anything the model saw in training — a caveat kriging's variance surface expresses differently (as a function of
  distance to the nearest stations rather than feature-space novelty).
- **`feature_importance`** — per time-step group, which UCP variable most reduced prediction error in the Random
  Forest. If building height or built-up fraction dominate, that's direct empirical support for the urban-canopy
  interpolation approach modeling the actual physical drivers of near-surface temperature, not just spatial
  proximity.
- **`n_stations`**, **`n_features`**, **`groups`** — bookkeeping: how many stations fed the model, how many UCP
  features were used (4 here, since GHSL-only), and which time steps/groups were successfully modeled.

Visualize the predicted surface with `lcz_plot_interp` (same plotting helper used for kriging output in notebook
04), and compare its texture — it should look "blockier" and more tied to urban-form boundaries than a kriging
surface, since the Random Forest prediction changes wherever the UCP values change, not smoothly with distance.

## `lcz_interp_eval_plus` — cross-validating the ML approach

`lcz_interp_eval_plus` reuses the exact same feature pipeline as `lcz_interp_map_plus`, but instead of predicting
across the full grid, it evaluates accuracy by leaving stations out (`LOOCV=True`, leave-one-station-out per time
step/group; or a random hold-out split via `split_ratio` when `LOOCV=False`) and comparing predicted vs. observed
temperature at the held-out locations. It returns a `pl.DataFrame` with columns
`station, group, observed, predicted, residual, rmse, mae, r2` — one row per held-out prediction, with the accuracy
columns letting you aggregate RMSE/MAE/R² however you like (overall, per station, per group).

We use the same coarse settings (`n_estimators=50`, GHSL-only UCPs) for consistency and speed, and the same
single-day date restriction as above so this cell finishes quickly. This is the apples-to-apples comparison point
against notebook 04's `lcz_interp_eval` (kriging LOOCV) — **run `lcz_interp_eval` from notebook 04 on the same
`df`/`map_path`, and compare its aggregate RMSE/MAE/R² against the one below.** Whichever method has the lower error
on *your* station network and *your* city's urban-form heterogeneity is the one to trust for a full-resolution
production map.

In [4]:
from LCZ4py.local.lcz_interp_map_plus import lcz_interp_eval_plus

eval_df = lcz_interp_eval_plus(
    map_path,
    df,
    var="airT",
    station_id="station",
    ml_model="rf",
    LOOCV=True,
    sp_res=1000.0,
    tp_res="day",
    n_estimators=50,
    process_ghsl=True,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=False,
    year=2019, month=1, day=1,
)

eval_df.head()

06:36:12 - LCZ4py.local.lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...


06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676


...

06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - Raster layers: 4


06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - DataFrame: 23 rows × 5 cols


06:36:12 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:36:12 - LCZ4py.local.lcz_interp_map_plus - INFO - Downloaded 4 / 4 urban parameter rasters.


station,group,observed,predicted,residual,cv_method,rmse,mae,r2
str,str,f64,f64,f64,str,f64,f64,f64
"""dahlem""","""2019-01-01 00:00:00""",6.35,6.472564,-0.122564,"""loocv""",0.220974,0.169195,0.039411
"""jagen91""","""2019-01-01 00:00:00""",6.311295,6.368196,-0.056901,"""loocv""",0.220974,0.169195,0.039411
"""albrecht""","""2019-01-01 00:00:00""",6.466154,6.394798,0.071356,"""loocv""",0.220974,0.169195,0.039411
"""dessauer""","""2019-01-01 00:00:00""",6.948127,6.371821,0.576305,"""loocv""",0.220974,0.169195,0.039411
"""bamberger""","""2019-01-01 00:00:00""",6.952638,6.747917,0.204721,"""loocv""",0.220974,0.169195,0.039411


### Interpreting the cross-validation output

Each row is one leave-one-station-out prediction: `observed` is the true station value, `predicted` is what the
Random Forest predicted for that station's location using UCPs from every *other* station's fitted model,
`residual = observed - predicted`. The `rmse`, `mae`, `r2` columns summarize accuracy (check whether the
implementation reports these per-group or as running aggregates — group by `group`/`station` in `eval_df` to compute
whatever aggregation you need, e.g. `eval_df.group_by("station").agg(pl.col("residual").abs().mean())` for
per-station MAE).

Compare this against notebook 04's `lcz_interp_eval(map_path, df, var="airT", station_id="station")` output on the
same data. A few interpretive patterns to look for:

- If RF+UCP has **lower** error than kriging, it suggests Berlin's station-to-station temperature differences are
  driven more by local urban form (impervious surface, building height) than by simple spatial proximity — exactly
  the situation this method is designed for.
- If kriging **wins**, it suggests either the station network is dense/well-distributed enough for smooth spatial
  interpolation to dominate, or the UCP feature set here (GHSL-only, no vegetation/roughness) is too coarse to
  capture what actually drives Berlin's temperature variation — worth revisiting with `process_vegetation=True`
  and/or `process_directional=True` for a richer (but slower) feature set.
- Either way, the comparison is the point: don't assume one method is universally better — measure it per dataset.

## Conclusion and next steps

This notebook introduced ML-based spatial interpolation as a UCP-driven alternative to kriging: `lcz_interp_map_plus`
trains a Random Forest (or OLS) on Urban Canopy Parameters extracted at station locations and predicts temperature
pixel-by-pixel from local urban form, rather than from distance-based spatial autocorrelation; `lcz_interp_eval_plus`
cross-validates that approach with the same LOOCV/hold-out logic as kriging's `lcz_interp_eval` (notebook 04),
enabling a direct, data-driven comparison between the two philosophies on your own station network.

**Previous**: [`04_spatial_interpolation_geostats`](04_spatial_interpolation_geostats.en.ipynb) — geostatistical
kriging interpolation.

**Next**: [`06_temporal_climate_metrics`](06_temporal_climate_metrics.en.ipynb) — deriving temporal climate metrics
(diurnal temperature range, degree hours) from station observations stratified by LCZ class.